In [42]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import time
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql.functions import udf

In [43]:
spark = SparkSession.builder \
    .appName("AdvanceSparkApp") \
    .getOrCreate()

In [44]:
df = spark.range(0, 1000000) \
          .withColumn("value", col("id") % 1000)

In [45]:
df.show(10)

+---+-----+
| id|value|
+---+-----+
|  0|    0|
|  1|    1|
|  2|    2|
|  3|    3|
|  4|    4|
|  5|    5|
|  6|    6|
|  7|    7|
|  8|    8|
|  9|    9|
+---+-----+
only showing top 10 rows


In [46]:
print("Before Partitions:", df.rdd.getNumPartitions())

Before Partitions: 2


In [47]:
df_repartitioned = df.repartition(10)

print("After Partitions:",
      df_repartitioned.rdd.getNumPartitions())

After Partitions: 10


In [48]:
df_coalesced = df_repartitioned.coalesce(2)

print("After Coalesced:",
      df_coalesced.rdd.getNumPartitions())

After Coalesced: 2


In [59]:
optimized_df = df.filter(col("value") > 500) \
                 .filter(col("id") < 5000)

In [60]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#182L, (id#182L % 1000) AS value#183L]
+- *(1) Filter (((id#182L % 1000) > 500) AND (id#182L < 5000))
   +- *(1) Range (0, 1000000, step=1, splits=2)




In [61]:
strat_time  =  time.time()
count_uncached = optimized_df.count()
end_time = time.time()
print(f"1.optimzed execution | count:{count_uncached} | Time:{round(end_time - start_time,4)} seconds")

1.optimzed execution | count:2495 | Time:32.4474 seconds


In [62]:
start_time = time.time()

count_uncached = optimized_df.count()

end_time = time.time()

print(
    f"UNCACHED Execution | "
    f"Count: {count_uncached} | "
    f"Time: {round(end_time-start_time,4)} sec"
)

UNCACHED Execution | Count: 2495 | Time: 0.0856 sec


In [63]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [64]:
start_time = time.time()

count_first_cache = optimized_df.count()

end_time = time.time()

print(
    f"FIRST CACHED Action | "
    f"Count: {count_first_cache} | "
    f"Time: {round(end_time-start_time,4)} sec"
)

FIRST CACHED Action | Count: 2495 | Time: 0.2862 sec


In [90]:
start_time = time.time()

count_second_cache = optimized_df.count()

end_time = time.time()

print(
    f"SECOND CACHED Action | "
    f"Count: {count_second_cache} | "
    f"Time: {round(end_time-start_time,4)} sec"
)

SECOND CACHED Action | Count: 2495 | Time: 0.2229 sec


In [91]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [92]:
#method usig python 
data = [("Alice",25),
        ("Bob",17),
        ("Charlie",35),
        ("David",15)]

df1 = spark.createDataFrame(data, ["Name","Age"])

df1.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 35|
|  David| 15|
+-------+---+



In [93]:
def categorize_age(age):
    if age >= 18:
        return "Adult"
    return "Minor"

In [94]:
categorize_age(25)

'Adult'

In [95]:
age_category_udf = udf(categorize_age,StringType())

In [96]:
df_method1 = df1.withColumn("category ",age_category_udf(col("Age")))
print("method 1 : Standard udf via Dataframe API")
df_method1.show()


method 1 : Standard udf via Dataframe API
+-------+---+---------+
|   Name|Age|category |
+-------+---+---------+
|  Alice| 25|    Adult|
|    Bob| 17|    Minor|
|Charlie| 35|    Adult|
|  David| 15|    Minor|
+-------+---+---------+



In [100]:
from pyspark.sql.types import StringType

spark.udf.register(
    "sql_categorize_age",
    categorize_age,
    StringType()
)

26/06/13 07:09:47 WARN SimpleFunctionRegistry: The function sql_categorize_age replaced a previously registered function.


<function __main__.categorize_age(age)>

In [101]:
#methos using sql 
df1.createOrReplaceTempView("people")

In [102]:
sql_df = spark.sql("""SELECT Name, Age,sql_categorize_age(Age) AS Category FROM people """)

In [103]:
SELECT
 Name,
 Age,
 CASE
   WHEN Age >= 18 THEN 'Adult'
   ELSE 'Minor'
 END
FROM people

IndentationError: unexpected indent (4107481883.py, line 2)

In [ ]:
sql_df.show()